In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import json

INPUT_DIR = '/kaggle/input/datasets/apr-dataset'
OUTPUT_DIR = '/kaggle/working'

TEST_PATH = f'{INPUT_DIR}/test_loop_filtered_verified.jsonl'
K8_10_PATH = f'{INPUT_DIR}/phase2_batch_results_k8_10.jsonl'

test_samples = []
with open(TEST_PATH) as f:
    for line in f:
        test_samples.append(json.loads(line))
print(f'Loaded {len(test_samples)} test samples')

print("K8-10 file exists:", os.path.exists(K8_10_PATH))

In [ ]:
results = []
with open(K8_10_PATH) as f:
    for line in f:
        results.append(json.loads(line))

print(f'Total entries: {len(results)}')
if results:
    print(f'Sample entry keys: {list(results[0].keys())}')
    history_lengths = [len(r.get('history', [])) for r in results]
    print(f'History lengths — min: {min(history_lengths)}, max: {max(history_lengths)}, avg: {sum(history_lengths)/len(history_lengths):.1f}')
    print()
    print('First entry (sample):')
    print(json.dumps(results[0], indent=2)[:800])

In [ ]:
from collections import Counter

total = len(results)
verdicts = Counter(r.get('final_verdict') for r in results)
print(f'Total: {total}')
print(f'Verdict breakdown: {dict(verdicts)}')

unique_problem_ids = len(set(r.get('problem_id') for r in results))
print(f'Unique problem_ids: {unique_problem_ids}')

In [ ]:
print(json.dumps(results[0], indent=2))

In [ ]:
import json, os, re, time, subprocess, zipfile, ast

INPUT_DIR = '/kaggle/input/datasets/apr-dataset'  # ADJUST if your path differs from Cell 1's listing
OUTPUT_DIR = '/kaggle/working'

KB_PATH = f'{INPUT_DIR}/pattern_knowledge_base_final.json'
TEST_PATH = f'{INPUT_DIR}/test_loop_filtered_verified.jsonl'
TEST_CASES_DIR = f'{INPUT_DIR}/public_test_cases/codenet/public_test_cases'
OUTPUT_PATH = f'{OUTPUT_DIR}/phase2_batch_results_k10.jsonl'

SPEEDUP_THRESHOLD = 1.10
MAX_ATTEMPTS = 10

with open(KB_PATH) as f:
    pattern_knowledge_base = json.load(f)
print(f'Loaded KB: {len(pattern_knowledge_base)} patterns')

test_samples = []
with open(TEST_PATH) as f:
    for line in f:
        test_samples.append(json.loads(line))
print(f'Loaded {len(test_samples)} test samples')

print('Test cases dir exists:', os.path.isdir(TEST_CASES_DIR))
print('Number of test-case problems:', len(os.listdir(TEST_CASES_DIR)) if os.path.isdir(TEST_CASES_DIR) else 'N/A')

In [ ]:
def safe_parse(code):
    try:
        return ast.parse(code)
    except SyntaxError:
        return None

def has_append_in_loop(tree):
    for node in ast.walk(tree):
        if isinstance(node, (ast.For, ast.While)):
            for sub in ast.walk(node):
                if isinstance(sub, ast.Call) and isinstance(sub.func, ast.Attribute) and sub.func.attr == 'append':
                    return True
    return False

def has_add_in_loop(tree):
    for node in ast.walk(tree):
        if isinstance(node, (ast.For, ast.While)):
            for sub in ast.walk(node):
                if isinstance(sub, ast.Call) and isinstance(sub.func, ast.Attribute) and sub.func.attr == 'add':
                    return True
    return False

def has_membership_or_index_in_loop(tree):
    for node in ast.walk(tree):
        if isinstance(node, (ast.For, ast.While)):
            for sub in ast.walk(node):
                if isinstance(sub, ast.Compare) and any(isinstance(op, ast.In) for op in sub.ops):
                    return True
                if isinstance(sub, ast.Call) and isinstance(sub.func, ast.Attribute) and sub.func.attr == 'index':
                    return True
    return False

def max_nesting_depth(tree):
    max_d = [0]
    def _walk(node, depth):
        if isinstance(node, (ast.For, ast.While)):
            depth += 1
            max_d[0] = max(max_d[0], depth)
        for child in ast.iter_child_nodes(node):
            _walk(child, depth)
    _walk(tree, 0)
    return max_d[0]

def count_same_level_loops(tree):
    count = [0]
    def _walk(node, inside_loop):
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.For, ast.While)):
                if not inside_loop:
                    count[0] += 1
                _walk(child, inside_loop=True)
            else:
                _walk(child, inside_loop=inside_loop)
    _walk(tree, False)
    return count[0]

def extract_structural_features(code):
    tree = safe_parse(code)
    if tree is None:
        return None
    return {
        "has_append_in_loop": has_append_in_loop(tree),
        "has_add_in_loop": has_add_in_loop(tree),
        "has_membership_or_index_in_loop": has_membership_or_index_in_loop(tree),
        "max_nesting_depth": max_nesting_depth(tree),
        "same_level_loop_count": count_same_level_loops(tree),
    }

print('Module 1 ready.')

In [ ]:
def detect_candidate_patterns(features):
    candidates = []
    if features.get("has_append_in_loop"):
        candidates.append("P4_Loop_to_Comprehension")
    if features.get("has_add_in_loop"):
        candidates.append("P5_Set_Membership_Optimization")
    if features.get("has_membership_or_index_in_loop"):
        candidates.append("P6_Dictionary_Lookup_Optimization")
    if features.get("max_nesting_depth", 0) >= 2:
        candidates.append("P1_Nested_Loop_Elimination")
    if features.get("same_level_loop_count", 0) > 1:
        candidates.append("P2_Loop_Reduction")
    if not candidates:
        candidates.append("P3_Internal_Optimization")
    return candidates

PATTERN_ID_MAP = {
    "P1_Nested_Loop_Elimination": "P1", "P2_Loop_Reduction": "P2",
    "P3_Internal_Optimization": "P3", "P4_Loop_to_Comprehension": "P4",
    "P5_Set_Membership_Optimization": "P5", "P6_Dictionary_Lookup_Optimization": "P6",
}

def retrieve_poi(final_pattern_key, kb):
    pattern_id = PATTERN_ID_MAP.get(final_pattern_key)
    if pattern_id is None or pattern_id not in kb:
        return {}, {}
    entry = kb[pattern_id]
    return entry.get("poi", {}), entry

print('Module 2 ready.')

In [ ]:
!pip install -U bitsandbytes -q

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

GEN_MODEL_NAME = "deepseek-ai/deepseek-coder-6.7b-instruct"
gen_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL_NAME, quantization_config=gen_bnb_config, device_map="auto")

def call_gen_llm(prompt, max_new_tokens=800):
    messages = [{"role": "user", "content": prompt}]
    inputs = gen_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(gen_model.device)
    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.3,
            do_sample=True, pad_token_id=gen_tokenizer.eos_token_id,
        )
    input_length = inputs['input_ids'].shape[1]
    return gen_tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

print('DeepSeek-Coder-6.7B-Instruct loaded.')

In [ ]:
def build_repair_prompt(pattern_name, poi, slow_code):
    activation_str = "\n".join(f"- {c}" for c in poi.get("activation_conditions", poi.get("when", [])))
    constraints_str = "\n".join(f"- {c}" for c in poi.get("constraints", []))
    return f"""You are an expert Python engineer specializing in performance optimization.

Apply the following optimization pattern to the slow code below.

Pattern: {pattern_name}
Trigger conditions:
{activation_str}
Transformation: {poi.get('transformation', poi.get('how', ''))}
Constraints:
{constraints_str}

Slow code:
{slow_code}

Return ONLY the optimized Python code. No markdown. No explanations."""

def extract_code_from_response(response):
    match = re.search(r"```(?:python)?\s*\n(.*?)```", response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()

print('Module 5 ready.')

In [ ]:
def load_test_cases(problem_id, test_cases_dir):
    problem_dir = os.path.join(test_cases_dir, problem_id)
    if not os.path.isdir(problem_dir):
        return []
    cases = []
    i = 0
    while True:
        input_path = os.path.join(problem_dir, f'input.{i}.txt')
        output_path = os.path.join(problem_dir, f'output.{i}.txt')
        if not os.path.exists(input_path):
            break
        with open(input_path) as f:
            stdin_data = f.read()
        with open(output_path) as f:
            expected_output = f.read().strip()
        cases.append({'stdin': stdin_data, 'expected': expected_output})
        i += 1
    return cases

def check_compiles(code):
    try:
        compile(code, '<repair>', 'exec')
        return True, None
    except SyntaxError as e:
        return False, str(e)

def run_with_timeout(code, stdin_data, timeout_sec=5):
    start = time.time()
    try:
        result = subprocess.run(['python3', '-c', code], input=stdin_data,
                                 capture_output=True, text=True, timeout=timeout_sec)
        elapsed = time.time() - start
        return result.stdout.strip(), result.returncode == 0, elapsed
    except subprocess.TimeoutExpired:
        return None, False, timeout_sec

def check_functional_correctness(code, test_cases):
    for case in test_cases:
        output, success, _ = run_with_timeout(code, case['stdin'])
        if not success or output != case['expected']:
            return False, {**case, 'actual_output': output if output is not None else '(crashed or timed out)'}
    return True, None

def measure_runtime(code, test_cases, repeats=3):
    total_time = 0
    for _ in range(repeats):
        for case in test_cases:
            _, success, elapsed = run_with_timeout(code, case['stdin'])
            if not success:
                return None
            total_time += elapsed
    return total_time / repeats

def validate_repair(slow_code, repaired_code, problem_id, test_cases_dir, speedup_threshold=SPEEDUP_THRESHOLD):
    test_cases = load_test_cases(problem_id, test_cases_dir)
    if not test_cases:
        return {'verdict': 'RETRY', 'reason': f'No test cases found for {problem_id}'}
    compiles, syntax_error = check_compiles(repaired_code)
    if not compiles:
        return {'verdict': 'RETRY', 'reason': 'compile_error', 'detail': syntax_error}
    correct, failing_case = check_functional_correctness(repaired_code, test_cases)
    if not correct:
        return {'verdict': 'RETRY', 'reason': 'test_failure', 'detail': failing_case}
    slow_runtime = measure_runtime(slow_code, test_cases)
    repaired_runtime = measure_runtime(repaired_code, test_cases)
    if slow_runtime is None or repaired_runtime is None or repaired_runtime <= 0:
        return {'verdict': 'RETRY', 'reason': 'runtime_measurement_failed'}
    speedup = slow_runtime / repaired_runtime
    if speedup >= speedup_threshold:
        return {'verdict': 'ACCEPT', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}
    else:
        return {'verdict': 'RETRY', 'reason': 'insufficient_speedup', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}

print('Module 6 ready.')

In [ ]:
def build_feedback_prompt(pattern_name, poi, slow_code, previous_code, validation_result):
    reason = validation_result.get('reason')
    if reason == 'compile_error':
        feedback = f"Your previous attempt had a syntax error: {validation_result.get('detail')}"
    elif reason == 'test_failure':
        case = validation_result.get('detail', {}) or {}
        feedback = (f"Your previous attempt failed a test case.\nInput:\n{case.get('stdin', '')}\n"
                     f"Expected output:\n{case.get('expected', '')}\n"
                     f"Your code's ACTUAL output:\n{case.get('actual_output', '')}\nFix the logic error.")
    elif reason == 'insufficient_speedup':
        feedback = (f"Your previous attempt was correct but NOT fast enough "
                     f"({validation_result.get('speedup')}x, need >= {SPEEDUP_THRESHOLD}x). "
                     f"Try a genuinely more efficient approach.")
    else:
        feedback = "Your previous attempt did not pass validation. Try a different approach."

    base_prompt = build_repair_prompt(pattern_name, poi, slow_code)
    return f"""{base_prompt}

Your previous attempt:
{previous_code}

Feedback on that attempt:
{feedback}

Generate a NEW, corrected version that addresses this feedback."""


def load_completed_keys(output_path):
    completed = set()
    if os.path.exists(output_path):
        with open(output_path) as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    key = rec.get('unique_key', rec.get('problem_id'))
                    if key:
                        completed.add(key)
                except Exception:
                    continue
    return completed

print('Feedback prompt + checkpoint helper ready.')

In [ ]:
def repair_sample(sample, kb, test_cases_dir, max_attempts=MAX_ATTEMPTS):
    problem_id = sample.get('problem_id')
    slow_code = sample.get('input', '')

    features = extract_structural_features(slow_code)
    if features is None:
        return {'problem_id': problem_id, 'final_verdict': 'PARSE_ERROR', 'candidates': [], 'history': []}

    candidates = detect_candidate_patterns(features)
    final_pattern = candidates[0] if candidates else 'P3_Internal_Optimization'
    poi, kb_entry = retrieve_poi(final_pattern, kb)

    try:
        prompt = build_repair_prompt(final_pattern, poi, slow_code)
        current_code = extract_code_from_response(call_gen_llm(prompt))
    except Exception as e:
        return {'problem_id': problem_id, 'final_verdict': 'PIPELINE_ERROR', 'error': f'initial generation failed: {e}',
                'candidates': candidates, 'detected_pattern': final_pattern, 'history': []}

    history = []
    for attempt in range(1, max_attempts + 1):
        try:
            result = validate_repair(slow_code, current_code, problem_id, test_cases_dir)
        except Exception as e:
            result = {'verdict': 'RETRY', 'reason': f'validation_error: {e}'}
        result['attempt'] = attempt
        history.append(result)

        if result.get('verdict') == 'ACCEPT':
            return {'problem_id': problem_id, 'final_verdict': 'ACCEPT', 'detected_pattern': final_pattern,
                    'candidates': candidates, 'num_attempts': attempt, 'history': history, 'final_code': current_code}
        if attempt == max_attempts:
            return {'problem_id': problem_id, 'final_verdict': 'RETRY_EXHAUSTED', 'detected_pattern': final_pattern,
                    'candidates': candidates, 'num_attempts': attempt, 'history': history, 'final_code': current_code}

        try:
            fb_prompt = build_feedback_prompt(final_pattern, poi, slow_code, current_code, result)
            current_code = extract_code_from_response(call_gen_llm(fb_prompt))
        except Exception as e:
            return {'problem_id': problem_id, 'final_verdict': 'RETRY_EXHAUSTED', 'detected_pattern': final_pattern,
                    'candidates': candidates, 'num_attempts': attempt, 'history': history, 'final_code': current_code,
                    'note': f'feedback generation failed at attempt {attempt+1}: {e}'}

    return {'problem_id': problem_id, 'final_verdict': 'RETRY_EXHAUSTED', 'candidates': candidates, 'history': history}

print('repair_sample ready (robust version).')

In [ ]:
K7_SOURCE_PATH = f'{INPUT_DIR}/phase2_batch_results_test.jsonl'  # your complete, existing k=7 result
K10_OUTPUT_PATH = f'{OUTPUT_DIR}/phase2_batch_results_k10_continued.jsonl'
EXTEND_TO = 10

old_results = {}
with open(K7_SOURCE_PATH) as f:
    for line in f:
        rec = json.loads(line)
        old_results[rec['problem_id']] = rec

print(f'Loaded {len(old_results)} existing k=7 records (unique problems)')
accepted_count = sum(1 for r in old_results.values() if r.get('final_verdict') == 'ACCEPT')
print(f'Already ACCEPTED at k=7: {accepted_count}')
print(f'RETRY_EXHAUSTED at k=7 (extending to k={EXTEND_TO}): {len(old_results) - accepted_count}')

seen = set()
unique_samples = []
for sample in test_samples:
    pid = sample.get('problem_id')
    if pid not in seen:
        seen.add(pid)
        unique_samples.append(sample)
print(f'Unique-problem sample list: {len(unique_samples)}')


In [ ]:
def continue_repair(old_record, sample, kb, test_cases_dir, extend_to=EXTEND_TO):
    if old_record.get('final_verdict') == 'ACCEPT':
        return old_record

    problem_id = sample.get('problem_id')
    slow_code = sample.get('input', '')
    final_pattern = old_record.get('detected_pattern', 'P3_Internal_Optimization')
    candidates = old_record.get('candidates', [])
    poi, kb_entry = retrieve_poi(final_pattern, kb)

    history = list(old_record.get('history', []))
    current_code = old_record.get('final_code', '')
    if not history or not current_code:
        return old_record

    last_result = history[-1]
    start_attempt = len(history) + 1

    for attempt in range(start_attempt, extend_to + 1):
        try:
            fb_prompt = build_feedback_prompt(final_pattern, poi, slow_code, current_code, last_result)
            current_code = extract_code_from_response(call_gen_llm(fb_prompt))
            result = validate_repair(slow_code, current_code, problem_id, test_cases_dir)
        except Exception as e:
            result = {'verdict': 'RETRY', 'reason': f'error: {e}'}
        result['attempt'] = attempt
        history.append(result)

        if result.get('verdict') == 'ACCEPT':
            return {'problem_id': problem_id, 'final_verdict': 'ACCEPT', 'detected_pattern': final_pattern,
                    'candidates': candidates, 'num_attempts': attempt, 'history': history, 'final_code': current_code}
        last_result = result

    return {'problem_id': problem_id, 'final_verdict': 'RETRY_EXHAUSTED', 'detected_pattern': final_pattern,
            'candidates': candidates, 'num_attempts': extend_to, 'history': history, 'final_code': current_code}

print('continue_repair ready.')

In [ ]:
completed = load_completed_keys(K10_OUTPUT_PATH)
print(f'Already processed: {len(completed)}')

total = len(unique_samples)
with open(K10_OUTPUT_PATH, 'a') as out_f:
    for i, sample in enumerate(unique_samples):
        problem_id = sample.get('problem_id')
        if problem_id in completed:
            continue
        old_record = old_results.get(problem_id)
        if old_record is None:
            continue

        if old_record.get('final_verdict') == 'ACCEPT':
            print(f'[{i+1}/{total}] {problem_id} -- already ACCEPTED at k=7, copying as-is')
            record = dict(old_record)
            record['unique_key'] = problem_id
            out_f.write(json.dumps(record) + '\n')
            out_f.flush()
            continue

        print(f'[{i+1}/{total}] {problem_id} -- extending past k=7...')
        record = continue_repair(old_record, sample, pattern_knowledge_base, TEST_CASES_DIR, extend_to=EXTEND_TO)
        record['unique_key'] = problem_id
        out_f.write(json.dumps(record) + '\n')
        out_f.flush()
        print(f'   -> {record["final_verdict"]}')

print('Continuation run complete.')

In [ ]:
results = []
with open(K10_OUTPUT_PATH) as f:
    for line in f:
        results.append(json.loads(line))

def score_at_k(results, k):
    total = len(results)
    opt_count = 0
    pass1_count = 0
    for r in results:
        history = r.get('history', [])[:k]
        if not history:
            continue
        if any(h.get('verdict') == 'ACCEPT' for h in history):
            opt_count += 1
        if history[0].get('verdict') == 'ACCEPT' or history[0].get('reason') == 'insufficient_speedup':
            pass1_count += 1
    print(f'k={k}: %OPT = {100*opt_count/total:.1f}%, Pass@1 = {100*pass1_count/total:.1f}%')

for k in range(1, 11):
    score_at_k(results, k)

In [ ]:
results_302 = []
with open(K7_SOURCE_PATH) as f:
    for line in f:
        results_302.append(json.loads(line))

print(f'Total lines: {len(results_302)}')
print(f'Unique problem_ids: {len(set(r["problem_id"] for r in results_302))}')

In [ ]:
from collections import Counter

pid_counts = Counter(r['problem_id'] for r in results_302)
# Find a problem_id that appears multiple times
dup_pid = [pid for pid, count in pid_counts.items() if count > 1][0]
print(f'Checking duplicate problem_id: {dup_pid} (appears {pid_counts[dup_pid]} times)')

dup_records = [r for r in results_302 if r['problem_id'] == dup_pid]
for i, r in enumerate(dup_records):
    print(f'\n--- Occurrence {i+1} ---')
    print(f'Verdict: {r.get("final_verdict")}')
    print(f'Final code (first 200 chars): {r.get("final_code", "")[:200]}')

In [ ]:
import statistics

def score_at_k_full(results, k):
    total = len(results)
    opt_count = 0
    pass1_count = 0
    speedups = []

    for r in results:
        history = r.get('history', [])[:k]
        if not history:
            continue
        accepted_in_k = [h for h in history if h.get('verdict') == 'ACCEPT']
        if accepted_in_k:
            opt_count += 1
            speedups.append(accepted_in_k[0]['speedup'])  # first ACCEPT within this k-window
        if history[0].get('verdict') == 'ACCEPT' or history[0].get('reason') == 'insufficient_speedup':
            pass1_count += 1

    median_sp = statistics.median(speedups) if speedups else None
    mean_sp = statistics.mean(speedups) if speedups else None

    print(f'k={k}: %OPT = {100*opt_count/total:.1f}%, Pass@1 = {100*pass1_count/total:.1f}%, '
          f'Median speedup = {median_sp:.3f}x' if median_sp else f'k={k}: no accepts', end='')
    if mean_sp:
        print(f', Mean speedup = {mean_sp:.3f}x')
    else:
        print()

for k in range(1, 11):
    score_at_k_full(results, k)

In [ ]:
from collections import Counter

pid_counts = Counter(r['problem_id'] for r in results_302)
dup_pid = [pid for pid, count in pid_counts.items() if count > 1][0]
print(f'Checking duplicate problem_id: {dup_pid} (appears {pid_counts[dup_pid]} times)')

dup_records = [r for r in results_302 if r['problem_id'] == dup_pid]
for i, r in enumerate(dup_records):
    print(f'\n--- Occurrence {i+1} ---')
    print(f'Verdict: {r.get("final_verdict")}')
    print(f'Final code (first 200 chars): {r.get("final_code", "")[:200]}')

In [ ]:
K10_302_OUTPUT_PATH = f'{OUTPUT_DIR}/phase2_batch_results_k10_302scope.jsonl'
EXTEND_TO = 10

# results_302 already loaded -- match by position (assumes same iteration order as test_samples)
print(f'results_302 length: {len(results_302)}, test_samples length: {len(test_samples)}')
assert len(results_302) == len(test_samples), "Length mismatch -- order-based matching won't be safe!"

accepted_count = sum(1 for r in results_302 if r.get('final_verdict') == 'ACCEPT')
print(f'Already ACCEPTED at k=7: {accepted_count}')
print(f'RETRY_EXHAUSTED at k=7 (extending to k={EXTEND_TO}): {len(results_302) - accepted_count}')

In [ ]:
completed = load_completed_keys(K10_302_OUTPUT_PATH)
print(f'Already processed: {len(completed)}')

total = len(test_samples)
with open(K10_302_OUTPUT_PATH, 'a') as out_f:
    for i, sample in enumerate(test_samples):
        problem_id = sample.get('problem_id')
        unique_key = f"{problem_id}__{i}"
        if unique_key in completed:
            continue

        old_record = results_302[i]  # matched by position

        if old_record.get('final_verdict') == 'ACCEPT':
            print(f'[{i+1}/{total}] {unique_key} -- already ACCEPTED at k=7, copying as-is')
            record = dict(old_record)
            record['unique_key'] = unique_key
            out_f.write(json.dumps(record) + '\n')
            out_f.flush()
            continue

        print(f'[{i+1}/{total}] {unique_key} -- extending past k=7...')
        record = continue_repair(old_record, sample, pattern_knowledge_base, TEST_CASES_DIR, extend_to=EXTEND_TO)
        record['unique_key'] = unique_key
        out_f.write(json.dumps(record) + '\n')
        out_f.flush()
        print(f'   -> {record["final_verdict"]}')

print('302-scope continuation run complete.')

In [ ]:
results_final = []
with open(K10_302_OUTPUT_PATH) as f:
    for line in f:
        results_final.append(json.loads(line))

print(f'Total: {len(results_final)}')

import statistics

def score_at_k_full(results, k):
    total = len(results)
    opt_count = 0
    pass1_count = 0
    speedups = []

    for r in results:
        history = r.get('history', [])[:k]
        if not history:
            continue
        accepted_in_k = [h for h in history if h.get('verdict') == 'ACCEPT']
        if accepted_in_k:
            opt_count += 1
            speedups.append(accepted_in_k[0]['speedup'])
        if history[0].get('verdict') == 'ACCEPT' or history[0].get('reason') == 'insufficient_speedup':
            pass1_count += 1

    median_sp = statistics.median(speedups) if speedups else None
    mean_sp = statistics.mean(speedups) if speedups else None

    line = f'k={k}: %OPT = {100*opt_count/total:.1f}%, Pass@1 = {100*pass1_count/total:.1f}%'
    if median_sp:
        line += f', Median speedup = {median_sp:.3f}x, Mean speedup = {mean_sp:.3f}x'
    print(line)

for k in range(1, 11):
    score_at_k_full(results_final, k)

In [ ]:
import os
print("File exists:", os.path.exists(K10_302_OUTPUT_PATH))
print("Line count:", sum(1 for _ in open(K10_302_OUTPUT_PATH)))

In [ ]:
from IPython.display import FileLink
FileLink(K10_302_OUTPUT_PATH)

In [ ]:
with open(K10_302_OUTPUT_PATH) as f:
    content = f.read()
print(len(content))  # koto character, eta check koro age